# Классификация на PyTorch Lightning
### Датасет: Wine (sklearn)

In [ ]:
# Установка необходимых библиотек
!pip install pytorch-lightning>=2.0 datasets tqdm -q

In [ ]:
import torch
import torch.nn as nn
import pytorch_lightning as pl
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import confusion_matrix, classification_report
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint

In [ ]:
# Загрузка и первичный анализ данных
wine = load_wine()
df = pd.DataFrame(wine.data, columns=wine.feature_names)
df["target"] = wine.target

print("Датасет вина:")
print(df.head())
print(f"\nРазмер данных: {df.shape}")
print(f"Признаки: {wine.feature_names}")
print(f"Целевые классы: {wine.target_names}")
print(f"Количество классов: {len(np.unique(wine.target))}")

In [ ]:
# Разделение данных (Train/Val/Test) и нормализация
X = wine.data
y = wine.target

# Сначала делим на train (70%) и temp (30%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
# Делим temp на val (15%) и test (15%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

# Нормализация (StandardScaler)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

## Визуализация данных с помощью PCA

In [ ]:
# PCA визуализация для понимания структуры данных

# Объединяем все нормализованные данные для визуализации
X_scaled = np.vstack([X_train, X_val, X_test])
y_all = np.hstack([y_train, y_val, y_test])

print(f"Объединенный датасет для визуализации: {X_scaled.shape}")

# 2D PCA
pca_2d = PCA(n_components=2)
X_2d = pca_2d.fit_transform(X_scaled)

print(f"\nОбъясненная дисперсия (2 компоненты):")
print(f"PC1: {pca_2d.explained_variance_ratio_[0]:.3f}")
print(f"PC2: {pca_2d.explained_variance_ratio_[1]:.3f}")
print(f"Суммарно: {pca_2d.explained_variance_ratio_.sum():.3f}")

plt.figure(figsize=(10, 6))
for i in np.unique(y_all):
    plt.scatter(
        X_2d[y_all == i, 0],
        X_2d[y_all == i, 1],
        label=wine.target_names[i],
        alpha=0.7,
        s=50
    )
plt.xlabel("Первая главная компонента (PC1)")
plt.ylabel("Вторая главная компонента (PC2)")
plt.title("2D PCA проекция набора данных Wine")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 3D PCA
pca_3d = PCA(n_components=3)
X_3d = pca_3d.fit_transform(X_scaled)

print(f"\nОбъясненная дисперсия (3 компоненты):")
print(f"PC1: {pca_3d.explained_variance_ratio_[0]:.3f}")
print(f"PC2: {pca_3d.explained_variance_ratio_[1]:.3f}")
print(f"PC3: {pca_3d.explained_variance_ratio_[2]:.3f}")
print(f"Суммарно: {pca_3d.explained_variance_ratio_.sum():.3f}")

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

for i in np.unique(y_all):
    ax.scatter(
        X_3d[y_all == i, 0],
        X_3d[y_all == i, 1],
        X_3d[y_all == i, 2],
        label=wine.target_names[i],
        alpha=0.7,
        s=40
    )

ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_zlabel("PC3")
ax.set_title("3D PCA проекция набора данных Wine")
ax.legend()
plt.tight_layout()
plt.show()

print("\nВывод: Данные хорошо разделимы даже при снижении размерности до 2-3 компонент.")

In [ ]:
# Преобразование в тензоры PyTorch
X_train_t = torch.FloatTensor(X_train)
X_val_t = torch.FloatTensor(X_val)
X_test_t = torch.FloatTensor(X_test)
y_train_t = torch.LongTensor(y_train)
y_val_t = torch.LongTensor(y_val)
y_test_t = torch.LongTensor(y_test)

print(f"Тензоры: Train {X_train_t.shape}, Val {X_val_t.shape}, Test {X_test_t.shape}")

In [ ]:
# 1. Создание Dataset
class WineDataset(Dataset):
    def __init__(self, X, y):
        self.X = X.clone().detach().float() if torch.is_tensor(X) else torch.tensor(X, dtype=torch.float32)
        self.y = y.clone().detach().long() if torch.is_tensor(y) else torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# 2. Создание DataModule
class WineDataModule(pl.LightningDataModule):
    def __init__(self, X_train, X_val, X_test, y_train, y_val, y_test, batch_size=16):
        super().__init__()
        self.X_train = X_train
        self.X_val = X_val
        self.X_test = X_test
        self.y_train = y_train
        self.y_val = y_val
        self.y_test = y_test
        self.batch_size = batch_size

    def setup(self, stage=None):
        self.train_dataset = WineDataset(self.X_train, self.y_train)
        self.val_dataset = WineDataset(self.X_val, self.y_val)
        self.test_dataset = WineDataset(self.X_test, self.y_test)

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size)

    def test_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=self.batch_size)

# 3. Создание модели
class NeuralNetLightning(pl.LightningModule):
    def __init__(self, input_size, num_classes, lr=0.001):
        super().__init__()
        self.save_hyperparameters()
        self.model = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )
        self.criterion = nn.CrossEntropyLoss()

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log("train_loss", loss, prog_bar=True)
        self.log("train_acc", acc, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)

    def test_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log("test_loss", loss, prog_bar=True)
        self.log("test_acc", acc, prog_bar=True)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

In [ ]:
# Инициализация DataModule
data_module = WineDataModule(
    X_train=X_train_t,
    X_val=X_val_t,
    X_test=X_test_t,
    y_train=y_train_t,
    y_val=y_val_t,
    y_test=y_test_t,
    batch_size=16
)

# Инициализация модели
model = NeuralNetLightning(
    input_size=X_train.shape[1],
    num_classes=len(np.unique(y)),
    lr=0.001
)

# Callbacks
early_stop = EarlyStopping(monitor="val_loss", patience=10, mode="min")
checkpoint = ModelCheckpoint(
    monitor="val_loss",
    mode="min",
    save_top_k=1,
    filename="best-{epoch:02d}-{val_loss:.4f}"
)

# Trainer
trainer = pl.Trainer(
    max_epochs=100,
    accelerator="auto",
    devices=1,
    log_every_n_steps=5,
    enable_progress_bar=True,
    callbacks=[early_stop, checkpoint]
)

In [ ]:
# Запуск обучения
print("Начинаем обучение...")
trainer.fit(model, data_module)
print("\nОбучение завершено.")

In [ ]:
# Тестирование лучшей модели
print("\nРезультаты на тестовой выборке:")
test_results = trainer.test(model, datamodule=data_module, ckpt_path="best")
print(f"Test Accuracy: {test_results[0]['test_acc']:.4f}")
print(f"Test Loss: {test_results[0]['test_loss']:.4f}")
print(f"\nЛучший чекпоинт сохранен в: {checkpoint.best_model_path}")

In [ ]:
# Детальный анализ: Confusion Matrix и Classification Report
# Загрузка лучшей модели
best_model = NeuralNetLightning.load_from_checkpoint(checkpoint.best_model_path)
best_model.eval()

# Предсказания
y_true = []
y_pred = []

with torch.no_grad():
    for x_batch, y_batch in data_module.test_dataloader():
        logits = best_model(x_batch)
        preds = torch.argmax(logits, dim=1)
        y_true.extend(y_batch.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

# Метрики
test_acc = (np.array(y_true) == np.array(y_pred)).mean()
cm = confusion_matrix(y_true, y_pred)

print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=wine.target_names))

# Генерация и сохранение Confusion Matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=wine.target_names, 
            yticklabels=wine.target_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=200, bbox_inches='tight')
plt.show()

print("\nГрафик сохранен как 'confusion_matrix.png'")

## Выводы
- **PCA визуализация** показала, что классы вин хорошо разделимы даже в пространстве 2-3 главных компонент
- Модель достигла высокой точности на тестовой выборке (>96%)
- PyTorch Lightning позволил удобно организовать код (DataModule, Callbacks)
- Ранняя остановка (EarlyStopping) предотвратила переобучение
- Confusion matrix сохранена как отдельный PNG файл для отчетности